# Overfitting — see it for yourself (NYC taxi fares)

A 30-second, self-contained demo of the single most important idea in machine learning:
a model can get **better and better on the data it trained on while getting *worse* on new data**.
That's overfitting.

We'll use a real sample of ~6,400 NYC taxi trips (built into the `seaborn` library, no download)
and try to predict the **fare** from the trip's **distance** and **passenger count**.

Just press the play button on each cell. ▶


In [ ]:
# Real NYC taxi trips — a ~6,400-ride sample built into seaborn (no download needed)
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

df = sns.load_dataset("taxis").dropna(subset=["distance", "passengers", "fare"])
print(f"{len(df)} taxi trips loaded")
df[["distance", "passengers", "fare"]].head()


### Train models of increasing complexity

A decision tree's `max_depth` controls how complex it can get. Low depth = simple.
High depth = it can carve the data into tiny pieces and memorise individual trips.

We measure quality with **R²**: 1.0 = perfect, 0.0 = no better than always guessing the
average fare. We check it on **training** trips (ones it learned from) and on **test** trips
(ones it has never seen).


In [ ]:
# Predict the FARE (a number -> regression) from the trip
X = df[["distance", "passengers"]]
y = df["fare"]

# Hold out 30% the model never sees during training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"{'depth':>5} | {'train R2':>8} | {'test R2':>8}")
for depth in [1, 2, 3, 5, 8, 12, None]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    train_r2 = model.score(X_train, y_train)   # on data it studied
    test_r2  = model.score(X_test, y_test)     # on NEW data
    print(f"{str(depth):>5} | {train_r2:>8.3f} | {test_r2:>8.3f}")


### Now picture it

Watch the two lines pull apart. Training R² keeps climbing toward a perfect 1.0.
Test R² rises, **peaks**, then **falls** — the moment it starts dropping is the moment the
model stops *learning the pattern* and starts *memorising the noise*.


In [ ]:
import matplotlib.pyplot as plt

depths = [1, 2, 3, 5, 8, 12, None]
xs, train_scores, test_scores = [], [], []
for depth in depths:
    m = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
    xs.append(depth if depth is not None else 20)   # show "None" as a deep tree
    train_scores.append(m.score(X_train, y_train))
    test_scores.append(m.score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(xs, train_scores, "o-", label="train R²  (data it studied)")
plt.plot(xs, test_scores,  "s-", label="test R²   (new, unseen data)")
plt.axvline(xs[test_scores.index(max(test_scores))], ls="--", c="gray", alpha=.6)
plt.xlabel("tree depth  (more complex →)")
plt.ylabel("R²")
plt.title("Overfitting: training keeps improving while test peaks, then drops")
plt.legend(); plt.grid(alpha=.3)
plt.show()


### The takeaway

The best model isn't the most complex one — it's the one at the **peak of the test line**.
Past that point, extra complexity makes the model look smarter on paper and perform worse in
reality.

**Try it yourself:** change `y = df["fare"]` to `y = df["tip"]` and re-run. Tips depend on
unpredictable human generosity, so the overfitting gets dramatic — the test R² can even go
*negative* (a model literally worse than guessing the average).
